# 1. Create and activate Virtaul Environment



In [ ]:
# !pip install virtualenv
# !virtualenv myenv_advanced-rag-with-llama-3-in-langchain
# !source .myenv_advanced-rag-with-llama-3-in-langchain/bin/activate

Der Befehl "source" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


# 2. Install libraries

In [1]:
!pip -qqq install pip --progress-bar off
!pip -qqq install langchain-groq==0.1.3 --progress-bar off
!pip -qqq install langchain==0.1.17 --progress-bar off
!pip -qqq install llama-parse==0.1.3 --progress-bar off
!pip -qqq install qdrant-client==1.9.1  --progress-bar off
!pip -qqq install "unstructured[md]"==0.13.6 --progress-bar off
!pip -qqq install fastembed==0.2.7 --progress-bar off
!pip -qqq install flashrank==0.2.4 --progress-bar off

# 3. Retrieval-Augmented Generation (RAG) Pipeline für Markdown-Dokumente

Dieses Notebook implementiert eine RAG-Pipeline zur Verarbeitung von Markdown-Dokumenten. Die Pipeline ermöglicht es, Dokumente zu laden, in Chunks zu teilen, Vektorisierungen zu erstellen, diese in einer Qdrant-Vektordatenbank zu speichern und präzise Antworten auf Fragen mithilfe eines Grok-Modells zu generieren.

## Hauptfunktionen:
- **Dokumentenverarbeitung**: Laden von Markdown-Dateien mit `UnstructuredMarkdownLoader`.
- **Text-Splitting**: Aufteilen der Dokumente in Chunks mit `RecursiveCharacterTextSplitter`.
- **Vektorisierung**: Erstellung von Embeddings mit `FastEmbedEmbeddings`.
- **Vektordatenbank**: Speicherung der Embeddings in `Qdrant` für effizientes Retrieval.
- **Reranking**: Verwendung von `FlashrankRerank` für kontextbezogenes Dokumenten-Ranking.
- **Q&A mit Grok**: Beantwortung von Fragen basierend auf den geladenen Dokumenten mit `ChatGroq`.
- **API-Schlüssel**: Sicherer Zugriff auf den Groq-API-Schlüssel über Colab Secrets.

## Voraussetzungen:
- Stelle sicher, dass der `GROQ_API_KEY` in den Colab Secrets gespeichert ist.
- Lade eine Markdown-Datei in Colab hoch (z. B. unter `/content/dein_dokument.md`).
- Installiere die erforderlichen Bibliotheken (siehe unten).

In [ ]:
import os
import textwrap
from pathlib import Path

# from google.colab import userdata
from IPython.display import Markdown
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

In [4]:
from llama_parse import LlamaParse

# Herunterladen der PDF-Dokumente für die RAG-Pipeline

Dieser Abschnitt erstellt ein Verzeichnis `data` in der Google Colab-Umgebung und lädt die Datei `meta-earnings.pdf` von Google Drive herunter. Diese Datei wird später in der Retrieval-Augmented Generation (RAG)-Pipeline verwendet, um Inhalte zu verarbeiten und Fragen basierend auf den Dokumenten zu beantworten.

## Schritte:
1. **Verzeichnis erstellen**: Erstellt ein Verzeichnis namens `data`, um die heruntergeladene Datei zu speichern.
2. **Datei herunterladen**: Lädt die Datei `meta-earnings.pdf` mit `gdown` von Google Drive (ID: `1ee-BhQiH-S9a2IkHiFbJz9eX_SfcZ5m9`) in das Verzeichnis `/content/data/`.

## Voraussetzungen:
- Die `gdown`-Bibliothek muss installiert sein. Führe die folgende Zelle aus, um sie zu installieren:
  ```bash
  !pip install gdown

In [5]:
!mkdir data
!gdown 1ee-BhQiH-S9a2IkHiFbJz9eX_SfcZ5m9 -O "data/meta-earnings.pdf"

Downloading...
From: https://drive.google.com/uc?id=1ee-BhQiH-S9a2IkHiFbJz9eX_SfcZ5m9
To: /content/data/meta-earnings.pdf
100% 160k/160k [00:00<00:00, 97.0MB/s]


## Document Parsing

# Parsen der Meta Q1 2024 Finanzergebnisse mit LlamaParse

Dieser Abschnitt verwendet `LlamaParse`, um die Datei `meta-earnings.pdf` zu verarbeiten, die die Finanzergebnisse von Meta für das erste Quartal 2024 enthält. Die Datei umfasst nicht geprüfte Finanzberichte, Management-Diskussionen, Analysen und SEC-spezifische Offenlegungen mit zahlreichen Tabellen. Der Parser wandelt die PDF-Datei in Markdown um, um die Inhalte für die nachfolgende RAG-Pipeline vorzubereiten.

## Schritte:
1. **Anweisung definieren**: Eine spezifische Parsing-Anweisung wird erstellt, um `LlamaParse` auf die Struktur der Finanzdaten und Tabellen hinzuweisen.
2. **Parser konfigurieren**: `LlamaParse` wird mit dem API-Schlüssel aus Colab Secrets, Markdown-Ausgabe, verbose Logging und einem Timeout von 5000 ms initialisiert.
3. **PDF laden**: Die Datei `/content/data/meta-earnings.pdf` wird asynchron geladen und in Markdown umgewandelt.

## Voraussetzungen:
- **LLAMA_CLOUD_API_KEY**: Stelle sicher, dass der LlamaCloud-API-Schlüssel in den Colab Secrets unter `LLAMA_PARSE` gespeichert ist.
- **PDF-Datei**: Die Datei `meta-earnings.pdf` muss im Verzeichnis `/content/data/` vorhanden sein (siehe vorherige Zelle zum Herunterladen).

In [8]:
instruction = """The provided document is Meta First Quarter 2024 Results.
This form provides detailed financial information about the company's performance for a specific quarter.
It includes unaudited financial statements, management discussion and analysis, and other relevant disclosures required by the SEC.
It contains many tables.
Try to be precise while answering the questions"""

parser = LlamaParse(
    api_key=userdata.get("LLAMA_PARSE"),
    result_type="markdown",
    parsing_instruction=instruction,
    verbose = True,
    max_timeout=5000,
)

llama_parse_documents = await parser.aload_data("./data/meta-earnings.pdf")

Started parsing the file under job_id 9da6cd91-5ea1-4958-96b1-47734b47c412


In [9]:
parsed_doc = llama_parse_documents[0]

In [10]:
Markdown(parsed_doc.text[:4096])

# Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta founder and CEO. "The new version of Meta AI with Llama 3 is another step towards building the world's leading AI. We're seeing healthy growth across our apps and we continue making steady progress building the metaverse as well."

# First Quarter 2024 Financial Highlights

| In millions, except percentages and per share amounts | 2024     | 2023     | % Change |
| ----------------------------------------------------- | -------- | -------- | -------- |
| Revenue                                               | $ 36,455 | $ 28,645 | 27 %     |
| Costs and expenses                                    | $ 22,637 | $ 21,418 | 6 %      |
| Income from operations                                | $ 13,818 | $ 7,227  | 91 %     |
| Operating margin                                      | 38 %     | 25 %     |          |
| Provision for income taxes                            | $ 1,814  | $ 1,598  | 14 %     |
| Effective tax rate                                    | 13 %     | 22 %     |          |
| Net income                                            | $ 12,369 | $ 5,709  | 117 %    |
| Diluted earnings per share (EPS)                      | $ 4.71   | $ 2.20   | 114 %    |

# First Quarter 2024 Operational and Other Financial Highlights

- Family daily active people (DAP) – DAP was 3.24 billion on average for March 2024, an increase of 7% year-over-year.
- Ad impressions – Ad impressions delivered across our Family of Apps increased by 20% year-over-year.
- Average price per ad – Average price per ad increased by 6% year-over-year.
- Revenue – Total revenue and revenue on a constant currency basis were $36.46 billion and $36.35 billion, respectively, both of which increased by 27% year-over-year.
- Costs and expenses – Total costs and expenses were $22.64 billion, an increase of 6% year-over-year.
- Capital expenditures – Capital expenditures, including principal payments on finance leases, were $6.72 billion.
- Capital return program – Share repurchases were $14.64 billion of our Class A common stock and dividends payments were $1.27 billion.
- Cash, cash equivalents, and marketable securities – Cash, cash equivalents, and marketable securities were $58.12 billion as of March 31, 2024. Free cash flow was $12.53 billion.
- Headcount – Headcount was 69,329 as of March 31, 2024, a decrease of 10% year-over-year.
---
# CFO Outlook Commentary

We expect second quarter 2024 total revenue to be in the range of $36.5-39 billion. Our guidance assumes foreign currency is a 1% headwind to year-over-year total revenue growth, based on current exchange rates.

We expect full-year 2024 total expenses to be in the range of $96-99 billion, updated from our prior outlook of $94-99 billion due to higher infrastructure and legal costs. For Reality Labs, we continue to expect operating losses to increase meaningfully year-over-year due to our ongoing product development efforts and our investments to further scale our ecosystem.

We anticipate our full-year 2024 capital expenditures will be in the range of $35-40 billion, increased from our prior range of $30-37 billion as we continue to accelerate our infrastructure investments to support our artificial intelligence (AI) roadmap.

While we are not providing guidance for years beyond 2024, we expect capital expenditures will continue to increase next year as we invest aggressively to support our ambitious AI research and product development efforts.

Absent any changes to our tax landscape, we expect our full-year 2024 tax rate to be in the mid-teens.

In addition, we continue to monitor an active regulatory landscape, including the increasing legal and regulatory headwinds in the EU and the U.S. that could significantly impact our business and our financial results.

Q1 was a good start to the year. We're seeing strong momentum within our Family 

In [11]:
document_path = Path("data/parsed_document.md")
with document_path.open("a") as f:
    f.write(parsed_doc.text)

## Vector Embeddings

In [15]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [16]:
loader = UnstructuredMarkdownLoader(document_path)
loaded_documents = loader.load()

In [17]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
docs = text_splitter.split_documents(loaded_documents)
len(docs)

11

In [18]:
print(docs[0].page_content)

Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta founder and CEO. "The new version of Meta AI with Llama 3 is another step towards building the world's leading AI. We're seeing healthy growth across our apps and we continue making steady progress building the metaverse as well."

First Quarter 2024 Financial Highlights

In millions, except percentages and per share amounts 2024 2023 % Change Revenue $ 36,455 $ 28,645 27 % Costs and expenses $ 22,637 $ 21,418 6 % Income from operations $ 13,818 $ 7,227 91 % Operating margin 38 % 25 % Provision for income taxes $ 1,814 $ 1,598 14 % Effective tax rate 13 % 22 % Net income $ 12,369 $ 5,709 117 % Diluted earnings per share (EPS) $ 4.71 $ 2.20 114 %

First Quarter 2024 Operational and Other Financial Highlights

Family daily active people (DA

In [ ]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

In [20]:
qdrant = Qdrant.from_documents(
    docs,
    embeddings,
    # location=":memory:",
    path="./db",
    collection_name="document_embeddings",
)

In [21]:
%%time
query = "What is the most important innovation from Meta?"
similar_docs = qdrant.similarity_search_with_score(query)

CPU times: user 141 ms, sys: 1.96 ms, total: 143 ms
Wall time: 141 ms


In [22]:
for doc, score in similar_docs:
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {score}")
    print("-" * 80)
    print()

text: Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta foun

score: 0.6171743278205013
--------------------------------------------------------------------------------

text: Webcast and Conference Call Information

Meta will host a conference call to discuss the results at 2:00 p.m. PT / 5:00 p.m. ET today. The live webcast of Meta's earnings conference call can be accessed at investor.fb.com, along with the earnings press rel

score: 0.5703891577704281
--------------------------------------------------------------------------------

text: META PLATFORMS, INC.

CONDENSED CONSOLIDATED BALANCE SHEETS

(In millions)

(Unaudited)

March 31, 2024 December 31, 2023 Assets Current assets: Cash and cash equivalents $ 32,307 $ 41,862 Marketable securities 25,813 23,541 Accounts receivable, net 

In [23]:
%%time
retriever = qdrant.as_retriever(search_kwargs={"k": 5})
retrieved_docs = retriever.invoke(query)

CPU times: user 144 ms, sys: 18.8 ms, total: 163 ms
Wall time: 162 ms


In [24]:
for doc in retrieved_docs:
    print(f"id: {doc.metadata['_id']}\n")
    print(f"text: {doc.page_content[:256]}\n")
    print("-" * 80)
    print()

id: 0d2d539b0d3a43a9bc6d7c3a7e13b360

text: Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta foun

--------------------------------------------------------------------------------

id: fa1e31faa84a48869853e8fb8289383b

text: Webcast and Conference Call Information

Meta will host a conference call to discuss the results at 2:00 p.m. PT / 5:00 p.m. ET today. The live webcast of Meta's earnings conference call can be accessed at investor.fb.com, along with the earnings press rel

--------------------------------------------------------------------------------

id: ab860ef1659d40fe9695b4526626c4c7

text: META PLATFORMS, INC.

CONDENSED CONSOLIDATED BALANCE SHEETS

(In millions)

(Unaudited)

March 31, 2024 December 31, 2023 Assets Current assets: Cash and cash equivalents $ 32,307 $ 41,862

## Reranking

In [25]:
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 122MiB/s] 


In [26]:
%%time
reranked_docs = compression_retriever.invoke(query)
len(reranked_docs)

Running pairwise ranking..
CPU times: user 2.56 s, sys: 48.8 ms, total: 2.61 s
Wall time: 3.37 s


3

In [27]:
for doc in reranked_docs:
    print(f"id: {doc.metadata['_id']}\n")
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {doc.metadata['relevance_score']}")
    print("-" * 80)
    print()

id: 0d2d539b0d3a43a9bc6d7c3a7e13b360

text: Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta foun

score: 0.1708654910326004
--------------------------------------------------------------------------------

id: fa1e31faa84a48869853e8fb8289383b

text: Webcast and Conference Call Information

Meta will host a conference call to discuss the results at 2:00 p.m. PT / 5:00 p.m. ET today. The live webcast of Meta's earnings conference call can be accessed at investor.fb.com, along with the earnings press rel

score: 0.009536861442029476
--------------------------------------------------------------------------------

id: ab860ef1659d40fe9695b4526626c4c7

text: META PLATFORMS, INC.

CONDENSED CONSOLIDATED BALANCE SHEETS

(In millions)

(Unaudited)

March 31, 2024 December 31, 2023 Assets Curre

## Q&A Over Document

In [28]:
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192")

In [29]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.

Responses should be properly formatted to be easily read.
"""

prompt = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [30]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True},
)

In [31]:
%%time
response = qa.invoke("What is the most significant innovation from Meta?")

Running pairwise ranking..


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: Meta Reports First Quarter 2024 Results

MENLO PARK, Calif. – April 24, 2024 – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the quarter ended March 31, 2024.

"It's been a good start to the year," said Mark Zuckerberg, Meta founder and CEO. "The new version of Meta AI with Llama 3 is another step towards building the world's leading AI. We're seeing healthy growth across our apps and we continue making steady progress building the metaverse as well."

First Quarter 2024 Financial Highlights

In millions, except percentages and per share amounts 2024 2023 % Change Revenue $ 36,455 $ 28,645 27 % Costs and expenses $ 22,637 $ 21,418 6 % Income from operations 

In [32]:
print_response(response)

Based on the provided information, I don't know what the most significant innovation from Meta is.
The text mentions the "new version of Meta AI with Llama 3" as a step towards building the world's
leading AI, but it does not provide further details about this innovation or its significance.

However, I can provide some additional information about Meta's recent performance based on the
provided financial highlights:

* Meta's revenue has increased by 27% compared to the same quarter last year, reaching $36.46
billion.
* The company's net income has also increased by 91% year-over-year, reaching $12.37 billion.
* Meta's family daily active people (DAP) has increased by 7% year-over-year, reaching 3.24 billion
on average for March 2024.
* The company's capital return program has resulted in $14.64 billion of share repurchases and $1.27
billion of dividend payments.


In [33]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": False},
)

In [34]:
%%time
response = qa.invoke("What is the revenue for 2024 and % change?")

Running pairwise ranking..
CPU times: user 1.94 s, sys: 5.19 ms, total: 1.94 s
Wall time: 2.49 s


In [35]:
Markdown(response["result"])

**Revenue for 2024 and % Change:**

* Revenue: $36,455 million
* Year-over-year change: 27%

**Additional Helpful Information:**

* Revenue excluding foreign exchange effect: $36,349 million
* Advertising revenue: $35,635 million
* Advertising revenue excluding foreign exchange effect: $35,530 million

In [36]:
%%time
response = qa.invoke("What is the revenue for 2023?")

Running pairwise ranking..
CPU times: user 1.91 s, sys: 3.17 ms, total: 1.91 s
Wall time: 2.5 s


In [37]:
print_response(response)

**Answer:** The revenue for 2023 is $28,645 million.

**Additional helpful information:**

* The revenue for 2024 is $36,455 million, which is a 27% increase from 2023.
* The revenue excluding foreign exchange effect for 2024 is $36,349 million, which is also a 27%
increase from 2023.


In [38]:
%%time
response = qa.invoke(
    "How much is the revenue minus the costs and expenses for 2024? Calculate the answer"
)

Running pairwise ranking..
CPU times: user 2.21 s, sys: 6.1 ms, total: 2.21 s
Wall time: 4.24 s


In [39]:
print_response(response)

**Answer:**
The revenue minus the costs and expenses for 2024 is $13,818 million.

**Additional Information:**
This amount is also referred to as the income from operations, and it represents a 91% year-over-
year increase.


In [40]:
%%time
response = qa.invoke(
    "How much is the revenue minus the costs and expenses for 2023? Calculate the answer"
)

Running pairwise ranking..
CPU times: user 1.95 s, sys: 5.09 ms, total: 1.96 s
Wall time: 2.86 s


In [41]:
print_response(response)

To calculate the revenue minus the costs and expenses, we need to find the revenue and costs and
expenses for 2023.

Revenue for 2023:
$28,645 (GAAP revenue)

Costs and Expenses for 2023:
We don't have a direct figure for costs and expenses. However, we can calculate the net income,
which is the revenue minus costs and expenses.

Net Income for 2023:
$5,709 (from the CONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS)

So, the revenue minus the costs and expenses for 2023 is equal to the net income:
$5,709

Additional helpful information:

* The revenue for 2024 is $36,455, which is a 27% increase from 2023.
* The company expects full-year 2024 total expenses to be in the range of $96-99 billion.
* The company's full-year 2024 capital expenditures will be in the range of $35-40 billion.


In [42]:
%%time
response = qa.invoke("What is the expected revenue for the second quarter of 2024?")

Running pairwise ranking..
CPU times: user 1.95 s, sys: 7.08 ms, total: 1.96 s
Wall time: 2.6 s


In [43]:
Markdown(response["result"])

**Answer:** The expected revenue for the second quarter of 2024 is in the range of $36.5-39 billion.

**Additional helpful information:**

* The guidance assumes foreign currency is a 1% headwind to year-over-year total revenue growth, based on current exchange rates.
* The first quarter 2024 revenue was $36.46 billion, which is a 27% year-over-year increase.

In [44]:
%%time
response = qa.invoke("What is the overall outlook of Q1 2024?")

Running pairwise ranking..
CPU times: user 1.97 s, sys: 3.96 ms, total: 1.98 s
Wall time: 2.91 s


In [45]:
print_response(response)

**Overall Outlook of Q1 2024:**

The overall outlook of Q1 2024 is positive. According to Mark Zuckerberg, Meta founder and CEO,
"It's been a good start to the year." The company is seeing strong momentum within its Family of
Apps and making important progress on its longer-term AI and Reality Labs initiatives.

**Key Highlights:**

* Revenue increased by 27% year-over-year to $36.46 billion.
* Net income increased by 117% year-over-year to $12.37 billion.
* Diluted earnings per share (EPS) increased by 114% year-over-year to $4.71.
* Family daily active people (DAP) increased by 7% year-over-year to 3.24 billion.
* Ad impressions delivered across the Family of Apps increased by 20% year-over-year.
* Average price per ad increased by 6% year-over-year.

These results indicate a good start to the year, with strong growth in revenue, net income, and
earnings per share, as well as increases in daily active people, ad impressions, and average price
per ad.


## References

- [Meta Reports First Quarter 2024 Results](https://s21.q4cdn.com/399680738/files/doc_financials/2024/q1/Meta-03-31-2024-Exhibit-99-1_FINAL.pdf)